# Image classification ViT written by Gemini

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from torchvision.transforms import Compose, ToTensor, Normalize


In [ ]:

# ==========================================
# 1. Patch Embedding
# ==========================================
class PatchEmbedding(nn.Module):
    """
    Splits the image into patches and projects them into the transformer's hidden dimension (d_model).
    We use a Conv2d layer with kernel_size and stride equal to the patch_size. 
    This is an efficient, mathematically equivalent way to slice and project the image.
    """
    def __init__(self, in_channels=3, patch_size=4, d_model=128):
        super().__init__()
        self.patch_size = patch_size
        self.projection = nn.Conv2d(
            in_channels, 
            d_model, 
            kernel_size=patch_size, 
            stride=patch_size
        )

    def forward(self, x):
        # x shape: (batch_size, channels, height, width)
        x = self.projection(x) 
        # New shape: (batch_size, d_model, grid_height, grid_width)
        
        # Flatten the spatial dimensions (grid_height * grid_width) into a sequence of patches
        x = x.flatten(2) 
        # New shape: (batch_size, d_model, num_patches)
        
        # Transpose to match Transformer expectation: (batch_size, num_patches, d_model)
        x = x.transpose(1, 2)
        return x


In [ ]:

# ==========================================
# 2. The Vision Transformer Model
# ==========================================
class VisionTransformer(nn.Module):
    def __init__(self, image_size=32, patch_size=4, in_channels=3, num_classes=10, 
                 d_model=128, nhead=4, num_layers=3, dim_feedforward=256, dropout=0.1):
        super().__init__()
        
        # 1. Patch Extractor & Embedder
        self.patch_embed = PatchEmbedding(in_channels, patch_size, d_model)
        
        # Calculate how many patches we will have
        num_patches = (image_size // patch_size) ** 2
        
        # 2. [CLS] Token: A special learnable vector prepended to the sequence.
        # Its final state will represent the entire image for classification.
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # 3. Positional Embedding: Unlike the fixed math formulas in text transformers,
        # ViTs usually use learnable positional parameters. +1 accounts for the CLS token.
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d_model))
        self.pos_drop = nn.Dropout(p=dropout)
        
        # 4. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 5. Classification Head
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        batch_size = x.shape[0]
        
        # Create patch embeddings
        x = self.patch_embed(x)  # Shape: (batch, num_patches, d_model)
        
        # Expand the [CLS] token to match the batch size and prepend it to the sequence
        cls_tokens = self.cls_token.expand(batch_size, -1, -1) 
        x = torch.cat((cls_tokens, x), dim=1) # Shape: (batch, num_patches + 1, d_model)
        
        # Add positional embeddings so the model knows where each patch came from
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        # Pass through the Transformer
        x = self.transformer_encoder(x)
        
        # Extract the [CLS] token's output (the 0th token in the sequence)
        cls_output = x[:, 0]
        
        # Pass through the final classification head
        logits = self.classifier(cls_output)
        return logits


In [ ]:

# ==========================================
# 3. Data Preparation (CIFAR-10)
# ==========================================
def prepare_data(batch_size=64):
    print("Loading CIFAR-10 dataset...")
    dataset = load_dataset("cifar10")
    
    # Define image transformations: Convert PIL image to PyTorch Tensor and normalize
    transform = Compose([
        ToTensor(),
        Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])
    
    # Apply transforms on the fly
    def apply_transforms(examples):
        # HuggingFace CIFAR-10 stores images under the key 'img'
        examples['pixel_values'] = [transform(img.convert("RGB")) for img in examples['img']]
        return examples
    
    dataset = dataset.with_transform(apply_transforms)
    
    # Custom collate function to batch the data correctly for PyTorch
    def collate_fn(batch):
        pixel_values = torch.stack([item['pixel_values'] for item in batch])
        labels = torch.tensor([item['label'] for item in batch])
        return {'pixel_values': pixel_values, 'labels': labels}

    train_loader = DataLoader(dataset["train"], shuffle=True, batch_size=batch_size, collate_fn=collate_fn)
    test_loader = DataLoader(dataset["test"], batch_size=batch_size, collate_fn=collate_fn)
    
    return train_loader, test_loader


In [ ]:

# ==========================================
# 4. Training Loop
# ==========================================
def train_model():
    # Hyperparameters
    batch_size = 64
    epochs = 5
    learning_rate = 3e-4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_loader, test_loader = prepare_data(batch_size)

    # Initialize Model
    # CIFAR-10 images are 32x32. Using patch_size=4 yields 8x8 = 64 patches.
    model = VisionTransformer(
        image_size=32, 
        patch_size=4, 
        num_classes=10, 
        d_model=128, 
        nhead=4, 
        num_layers=3
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    # Training
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for i, batch in enumerate(train_loader):
            images = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            if i % 100 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{i}/{len(train_loader)}], Loss: {loss.item():.4f}")
                
        print(f"--- Epoch {epoch+1} completed. Average Loss: {total_loss/len(train_loader):.4f} ---")

if __name__ == "__main__":
    train_model()